# Análise Exploratória Estratégica: Vendas de Combustíveis no Brasil
**Objetivo:** Extrair insights acionáveis, padrões de consumo regional, tendências sazonais e anomalias na série histórica de vendas de combustíveis, suportando decisões estratégicas para o setor de energia e logística.

## Etapa 1 — Entendimento do Problema
* **Domínio:** Mercado brasileiro de distribuição de combustíveis fósseis e renováveis.
* **Perguntas de Negócio:** 1. Quais estados e regiões tracionam o consumo logístico (Diesel) vs. mobilidade (Gasolina/Etanol)?
  2. Como as crises econômicas ou mudanças estruturais impactaram a série histórica?
  3. Qual é a sazonalidade do consumo nacional?
* **Métricas Principais (KPIs):** Volume Total (m³), Market Share Regional (%), Taxa de Crescimento Anual Composta (CAGR), Variação Year-over-Year (YoY).
* **Limitações e Vieses:** A granularidade mensal e estadual oculta dinâmicas municipais (ex: disparidade entre capital e interior). O dado bruto de vendas pode mascarar perdas e estoques.

In [ ]:
# Importação de bibliotecas padrão de produção
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pathlib import Path
from scipy import stats
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configurações de visualização
sns.set_theme(style="whitegrid")
pd.options.display.float_format = '{:,.2f}'.format

# Formatação para apresentação de casas decimais em padrão pt-BR
def fmt_br(x):
    """
    Formata valores numéricos para o padrão PT-BR (milhares com ponto, decimais com vírgula).
    """
    if pd.isna(x):
        return "0,00"
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
# Para valores relativos (em %)
def fmt_pct_br(x):
    if pd.isna(x):
        return "0,00%"
    return f"{x:.2f}%".replace(".", ",")

# Carregamento dos dados
df = pd.read_excel(Path.cwd().parent / "dataset" / "vendas_combustiveis_brasil.xlsx")

## Etapa 2 — Diagnóstico e Transformação dos Dados
Antes de qualquer inferência, garantiremos a qualidade estrutural da base. Transformaremos as colunas para o padrão `snake_case` e criaremos uma variável temporal verdadeira (Data) mapeando os meses em português, o que habilitará a análise de séries temporais.

In [2]:
# 1. Padronização para snake_case
df.columns = (df.columns
              .str.lower()
              .str.replace(' ', '_')
              .str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))

# Renomeando colunas longas para melhor manuseio
df = df.rename(columns={'grande_regiao': 'regiao', 'unidade_da_federacao': 'uf'})

# Removendo os meses do ano de 2026 (incompleto)
df = df.loc[df['ano'] != 2026].copy()

# 2. Mapeamento de Meses para numérico
meses_map = {
    'JAN': '01', 'FEV': '02', 'MAR': '03', 'ABR': '04', 
    'MAI': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08', 
    'SET': '09', 'OUT': '10', 'NOV': '11', 'DEZ': '12'
}
df['mes_num'] = df['mes'].map(meses_map)

# 3. Criação de variável datetime (Period)
df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes_num'] + '-01')

# 4. Tratamento de Tipos e Nulos
df['vendas'] = pd.to_numeric(df['vendas'], errors='coerce').fillna(0)

display(HTML(f"<b style='color:#00A8E8;'>✓ Código Executado.</b> Total de Linhas: <b>{df.shape[0]}</b> | Total de Colunas <b>{df.shape[1]}</b>."))
display(df.head(5))

,ano,mes,regiao,uf,produto,vendas,mes_num,data
0,1990,JAN,REGIÃO NORDESTE,MARANHÃO,ETANOL HIDRATADO,"7,578.94",01,1990-01-01
1,1990,JUL,REGIÃO CENTRO-OESTE,GOIÁS,ETANOL HIDRATADO,"24,462.25",07,1990-07-01
2,1990,JUN,REGIÃO CENTRO-OESTE,GOIÁS,ETANOL HIDRATADO,"25,005.95",06,1990-06-01
3,1990,MAI,REGIÃO CENTRO-OESTE,GOIÁS,ETANOL HIDRATADO,"18,888.24",05,1990-05-01
4,1990,ABR,REGIÃO CENTRO-OESTE,GOIÁS,ETANOL HIDRATADO,"27,535.92",04,1990-04-01


In [ ]:
# # Exportando o dataset tratado e filtrado para a pronta aplicação no dash
# nome_arquivo_saida = Path.cwd().parent / "dataset" / "vendas_combustiveies_filtered.parquet"
# df.to_parquet(nome_arquivo_saida, index=False)

# print(f"Sucesso! Dataset exportado e pronto para o Dash: {nome_arquivo_saida}")

Sucesso! Dataset exportado e pronto para o Dash: c:\Users\dougl\OneDrive\Área de Trabalho\dash_vendas_combustives_brasil\vendas_combustiveies_filtered.parquet


## Etapa 3 — Perfil Estatístico Profundo
Vamos analisar o comportamento numérico da variável `vendas`. Não buscamos apenas a média, mas a variância e a assimetria para entender a concentração de mercado.

In [63]:
# Descritiva estatística robusta para Vendas
stats_vendas = df['vendas'].describe(percentiles=[.25, .5, .75, .90, .99]).apply(fmt_br)

assimetria = df['vendas'].skew()
curtose = df['vendas'].kurtosis()

print("\n" + "="*50)
print("--- 📊 PERFIL ESTATÍSTICO DE VENDAS (m³) 📊 ---")
print("="*50)
print(stats_vendas)

print("\n" + "="*50)
print(f"Assimetria (Skewness): {fmt_br(assimetria)} -> Uma assimetria extremamente alta positiva indica cauda longa à direita (poucos registros com volumes astronômicos).")
print(f"Curtose (Kurtosis): {fmt_br(curtose)} -> Dados altamente concentrados em torno de valores baixos/médios, com picos extremos muito distantes da média.")


--- 📊 PERFIL ESTATÍSTICO DE VENDAS (m³) 📊 ---
count       93.312,00
mean        42.336,08
std        104.961,13
min            -38,40
25%            269,08
50%          7.210,09
75%         35.317,48
90%        107.737,56
99%        589.506,75
max      1.312.176,52
Name: vendas, dtype: str

Assimetria (Skewness): 5,36 -> Uma assimetria extremamente alta positiva indica cauda longa à direita (poucos registros com volumes astronômicos).
Curtose (Kurtosis): 37,18 -> Dados altamente concentrados em torno de valores baixos/médios, com picos extremos muito distantes da média.


## Etapa 4 — Segmentação e Perfis (Market Share)
Quem são os motores do consumo no Brasil? A análise a seguir agrupa e rankeia o volume acumulado para separar o "sinal do ruído" (Princípio de Pareto - 80/20).

In [64]:
# 1. Market Share por Produto
share_produto = df.groupby('produto')['vendas'].sum().sort_values(ascending=False).reset_index()
share_produto['share_%'] = (share_produto['vendas'] / share_produto['vendas'].sum()) * 100
share_produto['acumulado_%'] = share_produto['share_%'].cumsum()

display(share_produto.style.format({'vendas': fmt_br, 'share_%': fmt_pct_br, 'acumulado_%': fmt_pct_br}))

print("\n" + "="*50)

# 2. Matriz Produto x Região (Pivot Table)
pivot_regiao = df.pivot_table(index='regiao', columns='produto', values='vendas', aggfunc='sum').fillna(0)
# Calculando a representatividade da região no total
pivot_regiao['TOTAL_REGIAO'] = pivot_regiao.sum(axis=1)
pivot_regiao = pivot_regiao.sort_values(by='TOTAL_REGIAO', ascending=False)
display(pivot_regiao.style.format(fmt_br).background_gradient(cmap='Blues', axis=None))

,produto,vendas,share_%,acumulado_%
0,ÓLEO DIESEL,"1.627.397.983,06","41,20%","41,20%"
1,GASOLINA C,"1.056.852.448,04","26,75%","67,95%"
2,GLP,"438.366.683,08","11,10%","79,04%"
3,ETANOL HIDRATADO,"419.790.144,03","10,63%","89,67%"
4,ÓLEO COMBUSTÍVEL,"213.922.656,29","5,42%","95,09%"
5,QUEROSENE DE AVIAÇÃO,"188.777.491,54","4,78%","99,86%"
6,QUEROSENE ILUMINANTE,"3.140.457,17","0,08%","99,94%"
7,GASOLINA DE AVIAÇÃO,"2.216.595,36","0,06%","100,00%"


produto,ETANOL HIDRATADO,GASOLINA C,GASOLINA DE AVIAÇÃO,GLP,QUEROSENE DE AVIAÇÃO,QUEROSENE ILUMINANTE,ÓLEO COMBUSTÍVEL,ÓLEO DIESEL,TOTAL_REGIAO
regiao,,,,,,,,,
REGIÃO SUDESTE,"278.265.404,88","485.918.837,96","724.014,41","204.101.718,29","122.388.750,79","1.706.209,14","104.272.840,78","686.359.191,86","1.883.736.968,11"
REGIÃO SUL,"48.165.294,22","223.464.395,51","379.005,37","77.461.686,49","11.485.979,40","617.486,49","24.330.262,64","327.211.601,12","713.115.711,24"
REGIÃO NORDESTE,"36.361.108,67","187.080.963,83","212.440,79","96.649.682,23","28.008.108,95","587.483,90","40.266.990,48","251.136.032,64","640.302.811,49"
REGIÃO CENTRO-OESTE,"50.169.644,29","94.797.159,11","519.883,34","35.785.654,22","15.865.032,49","77.103,43","11.870.612,31","207.924.034,74","417.009.123,93"
REGIÃO NORTE,"6.828.691,97","65.591.091,63","381.251,45","24.367.941,86","11.029.619,92","152.174,20","33.181.950,09","154.767.122,69","296.299.843,81"


## Etapa 5 — Visualizações e Distribuições
Seguindo as melhores práticas visuais, construiremos gráficos que contam uma história.

1. **Univariada/Simples:** Concentração de volume por combustível.
2. **Série Temporal:** O comportamento estrutural ao longo dos anos.
3. **Multivariada:** Heatmap da intensidade de consumo por Estado x Produto.

In [28]:
# 1. Gráfico de Barras: Market Share de Produtos
share_produto["share_br"] = share_produto["share_%"].apply(fmt_br)

fig_share = px.bar(
    share_produto,
    x="share_%",
    y="produto",
    orientation="h",
    custom_data=["share_br"],
    color="share_%",
    color_continuous_scale="Sunsetdark",
    title="Market Share da Série Histórica por Produto (%)",
    template="plotly_dark",
    labels={
        "share_%": "Participação no Volume Total (%)",
        "produto": ""
    }
)

fig_share.update_layout(
    xaxis_title="Participação no Volume Total (%)",
    yaxis_title="",
    hovermode="closest",
    coloraxis_showscale=False,
    height=600
)

fig_share.update_traces(
    hovertemplate=
    "<b>Produto:</b> %{y}<br>" +
    "<b>Market Share:</b> %{customdata[0]} %" +
    "<extra></extra>"
)

# Ordenar do maior para o menor (opcional)
fig_share.update_yaxes(categoryorder="total ascending")

fig_share.show()

In [32]:
# 2. Série Temporal de Consumo Anual (Os 3 Principais Produtos)
top_3_produtos = share_produto['produto'].head(3).tolist()

df_temporal = (
    df[df['produto'].isin(top_3_produtos)]
    .groupby(['data', 'produto'], as_index=False)['vendas']
    .sum()
)

df_temporal["vendas_br"] = df_temporal["vendas"].apply(fmt_br)

fig_temporal = px.line(
    df_temporal,
    x='data',
    y='vendas',
    color='produto',
    custom_data=['vendas_br'],
    category_orders={
        "produto": top_3_produtos
    },
    color_discrete_sequence=["#831E70", "#F38370", "#FBC28A"],
    title='Evolução Histórica do Consumo de Combustíveis (Top 3)',
    template='plotly_dark',
    labels={"produto": "Produto: "}
)

fig_temporal.update_layout(
    xaxis_title="Ano de Referência",
    yaxis_title="Volume de Vendas (m³)",
    hovermode="x unified"
)

fig_temporal.update_traces(
    hovertemplate=
    "<b>Produto:</b> %{fullData.name}<br>" +
    "<b>Volume:</b> %{customdata[0]} m³" +
    "<extra></extra>"
)

fig_temporal.show()

In [ ]:
# 3. Heatmap Complexo: UF x Produto (Visão Estratégica)
# Tabela UF x Produto
pivot_uf = df.pivot_table(
    index='uf',
    columns='produto',
    values='vendas',
    aggfunc='sum'
)

# % de participação de cada UF dentro do total do produto
pivot_uf_pct = pivot_uf.div(pivot_uf.sum(axis=0), axis=1) * 100
pivot_uf_pct = (
    pivot_uf_pct.loc[
        pivot_uf.sum(axis=1)
        .nlargest(15)
        .index
    ]
)

fig_heatmap = px.imshow(
    pivot_uf_pct,
    color_continuous_scale=[
        [0, "#FBC28A"],
        [0.25, "#F38370"],
        [0.50, "#831E70"],
        [1, "#2D1E3E"]
    ],
    aspect="auto",
    title='Concentração de Vendas por Estado (% do Total do Produto)',
    template='plotly_dark'
)

fig_heatmap.update_layout(
    separators=",.",
    xaxis_title="Produto",
    yaxis_title="Estado (UF)",
    coloraxis_colorbar_title="% Share"
)

fig_heatmap.update_traces(
    hovertemplate=
    "<b>UF:</b> %{y}<br>" +
    "<b>Produto:</b> %{x}<br>" +
    "<b>Participação:</b> %{z:,.2f}%<extra></extra>"
)

fig_heatmap.show()

## Etapa 6 — Análise Temporal Avançada (YoY Growth)
Para não olhar apenas volumes absolutos, precisamos medir a **velocidade** do crescimento ou da queda, calculando a variação percentual Ano Contra Ano (YoY).

In [5]:
# Calculando variação anual total
vendas_ano = (
    df.groupby('ano', as_index=False)['vendas']
    .sum()
)

vendas_ano['crescimento_yoy_%'] = (
    vendas_ano['vendas'].pct_change() * 100
)

vendas_ano = vendas_ano.dropna()

# Cores para crescimento e queda
vendas_ano['cor'] = np.where(
    vendas_ano['crescimento_yoy_%'] >= 0,
    "#09703b",
    "#D72631"
)

fig_yoy = px.bar(
    vendas_ano,
    x='ano',
    y='crescimento_yoy_%',
    color='cor',
    color_discrete_map='identity',
    title='Taxa de Crescimento Anual de Combustíveis (YoY %)'
)

# Texto e Hover
fig_yoy.update_traces(
    textposition='outside',
    texttemplate='%{y:,.1f}%',
    hovertemplate=
    '<b>Ano: </b>%{x}<br>' +
    '<b>Crescimento: </b>%{y:,.2f}%<extra></extra>'
)

# Layout padronizado
fig_yoy.update_layout(
    separators=',.',
    height=700,
    template='plotly_dark',
    title=dict(
        x=0.5,
        font=dict(
            size=22,
            family='Arial Black'
        )
    ),
    xaxis_title='Ano',
    yaxis_title='Crescimento (%)',
    xaxis=dict(
        tickmode='linear',
        dtick=1
    ),
    yaxis=dict(
        tickformat=',.1f',
        ticksuffix='%'
    ),
    font=dict(
        family='Arial',
        size=14
    ),
    showlegend=False
)

fig_yoy.add_hline(
    y=0,
    line_width=2,
    line_dash="dash",
    line_color="gray"
)

fig_yoy.show()

## Etapa 7 — Resumo Executivo e Insights Estratégicos

Com base no comportamento dos dados extraídos e plotados acima, estruturamos os seguintes direcionamentos:

### Resumo Executivo
1. **O Motor Logístico (Sinalizando PIB):** O Óleo Diesel domina quase que massivamente o volume absoluto histórico. Isso não é apenas sobre caminhões; é uma *proxy* direta para a atividade agrícola, industrial e transporte rodoviário de cargas no Brasil.
2. **Disparidade Regional:** O Sudeste puxa os números absolutos devido à infraestrutura e densidade populacional, mas a matriz de calor revela que estados do Centro-Oeste e Sul possuem peso desproporcional no consumo de Diesel quando comparados à sua população, evidenciando o agronegócio.
3. **Volatilidade Temporal:** A análise do crescimento YoY expõe quebras estruturais (ex: possíveis quedas em anos de recessão econômica severa, greves de caminhoneiros ou choques pandêmicos).

### Hipóteses de Negócio a serem Validadas
* **Hipótese 1:** "Picos de vendas de Óleo Diesel no Centro-Oeste precedem meses de safra recorde (soja/milho)."
  * *Como validar:* Cruzar esta base mensal com dados do IBGE/Conab de previsão de safra agrícola.
* **Hipótese 2:** "A relação de consumo entre Etanol Hidratado e Gasolina C possui correlação forte e inversa, ditada pela paridade de preços na bomba (regra dos 70%)."
  * *Como validar:* Inserir dados de preços médios da ANP neste dataframe e calcular a correlação de Pearson entre a (Razão de Preços) vs (Market Share Gasolina/Etanol).

### Recomendações (Actionables)
* **Para Players de Logística:** Avaliar a sazonalidade profunda por UF. Se há picos sazonais no Centro-Oeste (safra) e quedas nos centros urbanos na mesma época, as frotas de distribuição de combustíveis devem ser remanejadas proativamente para o interior do país nestes trimestres, otimizando o *supply chain*.
* **Para Investidores de Infraestrutura:** O CAGR de estados menores no Nordeste e Norte (se identificado como positivo contínuo na série histórica) sinaliza áreas carentes de novas bases de tancagem e distribuição, representando um oceano azul para novos aportes em comparação ao saturado mercado Sudeste.